In [1]:
# ====================== CÀI (nếu thiếu) ======================
!pip install "transformers>=4.38" torch torchvision torchaudio scikit-learn pandas numpy tqdm vncorenlp underthesea cleanlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.7 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00

In [2]:
# ====================== CÀI BỔ SUNG (nếu thiếu) ======================
!pip install underthesea openpyxl -q

In [ ]:
# ====================== IMPORTS ======================
import os, re, math, warnings, numpy as np, pandas as pd, torch, torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ====================== CẤU HÌNH ======================
CKPT_PATH = "/kaggle/input/dulieu/best_phobert_multitask.pt"  

INPUT_FILES = [
    "/kaggle/input/dulieu/cafef_end(2014-2024).xlsx",
    "/kaggle/input/dulieu/tinnhanhchungkhoan_end(2014-2024).xlsx"
]  # sửa theo danh sách file của bạn

TEXT_COL_CANDIDATES = ["lọc", "loc", "text", "Lọc", "LOC", "Text"]  # ưu tiên 'lọc'
MAX_LENGTH = 256
BATCH_SIZE = 32

# (tuỳ chọn) file danh sách mã cổ phiếu để hỗ trợ mask ticker
TICKER_FILES = [
     "/kaggle/input/dulieu/Macp.xlsx",
]

# ====================== ENTITY MASKING (underthesea + ticker list) ======================
def load_ticker_set():
    tickers = set()
    for p in TICKER_FILES:
        if os.path.exists(p):
            try:
                df_t = pd.read_excel(p)
                for col in df_t.columns:
                    s = df_t[col]
                    vals = s.dropna().astype(str).str.strip().unique().tolist()
                    for v in vals:
                        if re.fullmatch(r"[A-Z]{2,6}", v):
                            tickers.add(v)
            except Exception as e:
                print(" Không đọc được", p, e)
    if tickers:
        print(f"Loaded {len(tickers)} tickers từ file.")
    return tickers

TICKER_SET = load_ticker_set()

def build_ticker_regex(ticker_set):
    if not ticker_set: return None
    escaped = [re.escape(t) for t in sorted(ticker_set, key=len, reverse=True)]
    pat = r"\b(" + "|".join(escaped) + r")\b"
    return re.compile(pat)

TICKER_RE = build_ticker_regex(TICKER_SET)

# underthesea NER (nếu có)
try:
    from underthesea import ner as uts_ner
    UTS_OK = True
except Exception:
    print(" underthesea chưa có, chỉ mask ticker in HOA nếu có.")
    UTS_OK = False

def mask_entities(text: str) -> str:
    if not isinstance(text, str) or not text: return ""
    s = text
    # 1) che mã CP theo danh sách (nếu có)
    if TICKER_RE is not None:
        s = TICKER_RE.sub("<NAME>", s)
    # 2) NER bằng underthesea (nếu sẵn sàng)
    if UTS_OK:
        try:
            tags = uts_ner(s)  # list[(token, pos, chunk, ner)]
            out_tokens = []
            i = 0
            while i < len(tags):
                token = tags[i][0]
                ner_tag = str(tags[i][-1])
                if ner_tag.startswith("B-"):
                    ent_type = ner_tag[2:]  # PER/ORG/LOC...
                    j = i + 1
                    while j < len(tags) and str(tags[j][-1]).startswith("I-"):
                        j += 1
                    if ent_type in ("PER","ORG"):
                        out_tokens.append("<NAME>")
                    elif ent_type == "LOC":
                        out_tokens.append("<LOC>")
                    else:
                        out_tokens.append(token)
                    i = j
                else:
                    # fallback: token in HOA 2-6 ký tự → <NAME>
                    if re.fullmatch(r"[A-Z]{2,6}", token):
                        out_tokens.append("<NAME>")
                    else:
                        out_tokens.append(token)
                    i += 1
            s = " ".join(out_tokens)
        except Exception:
            pass
    return s

# ====================== WORD SEGMENT (VnCoreNLP) ======================
def get_vncorenlp():
    try:
        from vncorenlp import VnCoreNLP
        jar_candidates = [
            "vncorenlp/VnCoreNLP-1.1.1.jar",
            "/content/VnCoreNLP/VnCoreNLP-1.1.1.jar",
            "/kaggle/working/VnCoreNLP-1.1.1.jar",
        ]
        jar = next((p for p in jar_candidates if os.path.exists(p)), None)
        if jar is None:
            print(" Không tìm thấy VnCoreNLP jar → bỏ qua tách từ (sẽ dùng nguyên văn).")
            return None
        return VnCoreNLP(jar, annotators="wseg", max_heap_size='-Xmx2g')
    except Exception as e:
        print(" Không khởi tạo được VnCoreNLP:", e)
        return None

RDR = get_vncorenlp()

def vn_word_segment(rdr, s: str) -> str:
    if rdr is None: return s
    try:
        sents = rdr.tokenize(s)
        return " ".join(" ".join(tokens) for tokens in sents)
    except Exception:
        return s

# ====================== PREPROCESS (đúng pipeline lúc train) ======================
def preprocess_text(s: str) -> str:
    s = mask_entities(s)
    s = vn_word_segment(RDR, s)
    return s

# ====================== DATASET CHO INFER ======================
class NewsDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tok = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

# ====================== MÔ HÌNH (2 biến thể để load ) ======================
class PhoBertMultiHeadMeanMLP(nn.Module):
    """Mean-pooling + MLP (nếu bạn đã train bản này)."""
    def __init__(self, model_name, num_labels=5, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        h = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.mlp = nn.Sequential(
            nn.Linear(h, h//2), nn.GELU(), nn.Dropout(0.1),
        )
        self.head_sent = nn.Linear(h//2, num_labels)
        self.head_risk = nn.Linear(h//2, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        x = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (x * mask).sum(1) / mask.sum(1).clamp_min(1e-6)  # mean pooling
        pooled = self.dropout(pooled)
        pooled = self.mlp(pooled)
        return self.head_sent(pooled), self.head_risk(pooled)

class PhoBertMultiHeadCLS(nn.Module):
    """[CLS]-token head (nếu bạn train bản cũ)."""
    def __init__(self, model_name, num_labels=5, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        h = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head_sent = nn.Linear(h, num_labels)
        self.head_risk = nn.Linear(h, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:,0,:]
        pooled = self.dropout(pooled)
        return self.head_sent(pooled), self.head_risk(pooled)

# ====================== LOAD CHECKPOINT ======================
def load_checkpoint_model(ckpt_path):
    assert os.path.exists(ckpt_path), f"Không tìm thấy checkpoint: {ckpt_path}"
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if not isinstance(ckpt, dict):
        raise TypeError("Checkpoint không hợp lệ (mong đợi dict).")

    if "model_name" not in ckpt:
        raise KeyError("Checkpoint thiếu khoá 'model_name'. Vui lòng lưu lại checkpoint có 'model_name'.")

    model_name = ckpt["model_name"]
    state = ckpt["state_dict"] if "state_dict" in ckpt else ckpt

    # thử bản mean+MLP trước, nếu sai hình dạng thì fallback CLS
    try:
        model = PhoBertMultiHeadMeanMLP(model_name=model_name).to(device)
        model.load_state_dict(state, strict=True)
        variant = "mean+MLP"
    except Exception:
        model = PhoBertMultiHeadCLS(model_name=model_name).to(device)
        model.load_state_dict(state, strict=True)
        variant = "[CLS]"

    tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    print(f"✓ Loaded model: {model_name} ({variant})")
    return model, tok

# ====================== PREDICT (có thanh tiến trình) ======================
@torch.no_grad()
def predict_multitask(model, loader):
    model.eval()
    s_idx_all, s_cf_all, r_idx_all, r_cf_all = [], [], [], []
    for b in tqdm(loader, desc="Scoring", leave=False):
        ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
        ls, lr = model(ids, mask)
        ps = torch.softmax(ls, -1); pr = torch.softmax(lr, -1)
        s_conf, s_idx = ps.max(-1); r_conf, r_idx = pr.max(-1)
        s_idx_all.append((s_idx+1).cpu().numpy())
        s_cf_all.append(s_conf.cpu().numpy())
        r_idx_all.append((r_idx+1).cpu().numpy())
        r_cf_all.append(r_conf.cpu().numpy())
    s_idx = np.concatenate(s_idx_all); s_cf = np.concatenate(s_cf_all)
    r_idx = np.concatenate(r_idx_all); r_cf = np.concatenate(r_cf_all)
    return s_idx, s_cf, r_idx, r_cf

# ====================== TÌM CỘT VĂN BẢN ======================
def find_text_col(df):
    for name in TEXT_COL_CANDIDATES:
        if name in df.columns: return name
    lowers = {str(c).lower(): c for c in df.columns}
    for name in [n.lower() for n in TEXT_COL_CANDIDATES]:
        if name in lowers: return lowers[name]
    raise ValueError(f"Không thấy cột văn bản (ưu tiên 'lọc'). Columns: {list(df.columns)}")

# ====================== CHỌN THƯ MỤC GHI ======================
def pick_writable_outdir(prefer_from_path):
    d = os.path.dirname(prefer_from_path) or "."
    test_path = os.path.join(d, "__write_test.tmp")
    try:
        with open(test_path, "wb") as f: f.write(b"ok")
        os.remove(test_path)
        return d
    except Exception:
        pass
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working"
    if os.path.isdir("/content"):
        return "/content"
    return "."

# ====================== SCORE 1 FILE EXCEL (có thanh tiến trình) ======================
def score_excel(path, model, tokenizer, sheet_name=0):
    print(f"\n=== Scoring: {os.path.basename(path)} ===")
    df = pd.read_excel(path, sheet_name=sheet_name)
    text_col = find_text_col(df)
    raw_texts = df[text_col].astype(str).fillna("").tolist()

    texts = []
    for t in tqdm(raw_texts, desc="Preprocess", leave=False):
        texts.append(preprocess_text(t))

    ds = NewsDataset(texts, tokenizer, max_length=MAX_LENGTH)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

    s_pred, s_conf, r_pred, r_conf = predict_multitask(model, dl)

    df_out = df.copy()
    df_out["llm_sentiment"] = s_pred.astype(int)
    df_out["llm_risk"]      = r_pred.astype(int)
    # df_out["conf_sentiment"] = np.round(s_conf, 4)
    # df_out["conf_risk"]      = np.round(r_conf, 4)

    out_dir = pick_writable_outdir(path)
    base = os.path.splitext(os.path.basename(path))[0] + "_scored.xlsx"
    out_path = os.path.join(out_dir, base)
    df_out.to_excel(out_path, index=False)
    print("✓ Đã ghi:", out_path)
    return out_path

# ====================== CHẠY ======================
if __name__ == "__main__":
    model, tokenizer = load_checkpoint_model(CKPT_PATH)
    print("✓ Loaded checkpoint:", CKPT_PATH)

    for p in INPUT_FILES:
        assert os.path.exists(p), f"Không thấy file: {p}"

    for p in INPUT_FILES:
        score_excel(p, model, tokenizer)


Device: cuda
Loaded 1450 tickers từ file.
 Không tìm thấy VnCoreNLP jar → bỏ qua tách từ (sẽ dùng nguyên văn).


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

2025-09-04 10:17:51.992563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756981072.347527      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756981072.447905      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Loaded model: vinai/phobert-large ([CLS])
✓ Loaded checkpoint: /kaggle/input/dulieu/best_phobert_multitask.pt

=== Scoring: cafef_end(2014-2024).xlsx ===


Scoring:  82%|████████▏ | 240/292 [05:20<01:09,  1.33s/it]     